## Identifying co-expressed genes in two different cell types

Some key assumptions and considerations:
1. Cells from one same cell type in one sample or brain region should be aliked. 
2. Asusing snRNA-seq data, we do not care/have the spatial neighbohood information here.
3. Due to high number of genes and combinations, only focusing on genes expressed in specific cell types. For example, only focus on genes in astrotyes and capillary. 
4. After identification of the correlated genes, filtering the results based on the DEG results for each cell type across brain regions. 

Actually, I can also generate a pseudobulk gene expression matrix for the some other analyses. 

In [ ]:
### Loading libries for analysis
suppressPackageStartupMessages({
    library(Seurat)
    library(dplyr)
    library(Matrix)
    library(tidyr)
    library(stringr)
    library(DESeq2)
    # library(WGCNA)
    library(Hmisc)
    
    library(ggplot2)
    library(colorRamp2)
    library(ComplexHeatmap)
    library(tibble)
    library(Mfuzz)
    })

In [ ]:
## set working dir
dir = "/home/kibr/Work/Vascular_atlas"
setwd(dir)

In [ ]:
## load the seurat object
h5ad_file="./Results/Revision_2/clean_object_revision_04242026.h5ad"
clean = schard::h5ad2seurat(h5ad_file)
table(clean$Cell_class)

In [ ]:
features <- clean[['RNA']][[]]
ribosomal_genes <- rownames(features[grep(pattern = "^RP[SL]", x = rownames(features)),])
mt_genes <- rownames(features[grep(pattern = "MT-", x = rownames(features)),])

genes_excluded <- c(ribosomal_genes, mt_genes)
GENE_KEEP = setdiff(Features(clean[["RNA"]]),genes_excluded)
length(GENE_KEEP)

# creating the new layer 
DefaultAssay(clean) <- "RNA"
table(rownames(clean) %in% GENE_KEEP)
temp.assay <- subset(clean[["RNA"]], features = GENE_KEEP)
Key(temp.assay) <- "rna_"
clean[["CLEAN"]] <- temp.assay

In [ ]:
# Columns containing sample ID and cell type info
sample_col <- "individualID"  
celltype_col <- "Cell_class"  

# Choose the assay and slot
assay_use <- "CLEAN"
slot_use <- "data"  # use "counts" if raw counts are preferred

In [ ]:
## merge the layers and normalizing the data
clean <- NormalizeData(clean, assay = assay_use, slot = slot_use, normalization.method = "LogNormalize", scale.factor = 10000)

print(clean)

In [ ]:
# Function to create pseudobulk matrix for a single cell type
CreatePseudoBulk <- function(seurat_obj, target_celltype, assay){
    seurat_sub <- subset(seurat_obj, subset = !!sym(celltype_col) == target_celltype)
    exp_mat <- AggregateExpression(object = seurat_sub,assays = assay,group.by = c("individualID","brain_region"))$CLEAN

    # For raw counts
    keep_genes <- rowSums(exp_mat > 0) >= 5
    exp_mat <- exp_mat[keep_genes, ]
    return(exp_mat)
}

## preparing vsd matrix
CreateVSTMatrix <- function(mat= mat,var.per=0.95){
    sample_metadata = as.data.frame(colnames(mat))
    sample_metadata$individualID = str_split_fixed(sample_metadata[,1],pattern = "_",n = 2)[,1]
    sample_metadata$brain_region = str_split_fixed(sample_metadata[,1],pattern = "_",n = 2)[,2]

    dds = DESeqDataSetFromMatrix(countData = mat,
                                  colData = sample_metadata,
                                  design = ~1)
    dds = estimateSizeFactors(dds,type = "poscounts")           
    vsd = vst(dds, blind = TRUE)
    # vsd = rlog(dds, blind = TRUE)
    vst_matrix = assay(vsd)

    ## only keeping the highly variable genes
    row_var <- rowVars(vst_matrix)
    q95_var <- quantile(row_var, var.per)
    vst_matrix <- vst_matrix[row_var> q95_var, ]

    return(vst_matrix)
}

CreateLogNormMatrix <- function(mat, var.per = 0.05, pseudocount = 1) {
  # Extract sample metadata from column names
  sample_metadata <- data.frame(sample = colnames(mat))
  sample_metadata$individualID <- str_split_fixed(sample_metadata$sample, "_", 2)[,1]
  sample_metadata$brain_region <- str_split_fixed(sample_metadata$sample, "_", 2)[,2]

  # # Perform log normalization (log1p or log2(CPM + 1))
  # # Option 1: log1p of raw counts (default)
  # log_mat <- log1p(mat)

  # Option 2 (if you prefer log2 CPM):
  library(edgeR)
  dge <- DGEList(counts = mat)
  dge <- calcNormFactors(dge)
  log_mat <- cpm(dge, log = TRUE, prior.count = pseudocount)

  # Filter for variable genes
  row_var <- rowVars(log_mat)
  var_threshold <- quantile(row_var, var.per)
  log_mat <- log_mat[row_var > var_threshold, ]

  return(log_mat)
}

In [ ]:
run_correlation = function(ct_1 = ct_1, ct_2 = ct_2){
    mat_1 = CreatePseudoBulk(seurat_obj = clean, target_celltype = ct_1, assay = "CLEAN")
    mat_2 = CreatePseudoBulk(seurat_obj = clean, target_celltype = ct_2, assay = "CLEAN")
    
    # Align on common samples
    common_samples <- intersect(colnames(mat_1), colnames(mat_2))
    mat_1 <- mat_1[, common_samples]
    mat_2 <- mat_2[, common_samples]

    print(dim(mat_1))
    print(dim(mat_2))

    vst_mat_1 = CreateVSTMatrix(mat = mat_1,var.per = 0.95)
    vst_mat_2 = CreateVSTMatrix(mat = mat_2,var.per = 0.95)

    rownames(vst_mat_1) <- paste0(rownames(vst_mat_1), "_A")
    rownames(vst_mat_2) <- paste0(rownames(vst_mat_2), "_B")

    ## Running the spearman correlation analysis on the two gene lists from two cell types. 
    combined <- cbind(t(vst_mat_1), t(vst_mat_2))  # samples × genes

    # rcorr() computes correlations and p-values
    result <- rcorr(as.matrix(combined), type = "spearman")  # or "pearson"

    # Extract relevant correlation block (astro × capillary only)
    n_1 <- nrow(vst_mat_1)
    n_2 <- nrow(vst_mat_2)

    cor_mat <- result$r[1:n_1, (n_1 + 1):(n_1 + n_2)]
    p_mat   <- result$P[1:n_1, (n_1 + 1):(n_1 + n_2)]

    rownames(cor_mat) = gsub("_A","",rownames(cor_mat))
    colnames(cor_mat) = gsub("_B","",colnames(cor_mat))

    rownames(p_mat) = gsub("_A","",rownames(p_mat))
    colnames(p_mat) = gsub("_B","",colnames(p_mat))
    # Flatten to vector
    pvals <- as.vector(p_mat)

    # Adjust p-values (FDR)
    padj <- p.adjust(pvals, method = "bonferroni")

    # Reshape to original matrix dimensions
    padj_mat <- matrix(padj, nrow = nrow(p_mat), ncol = ncol(p_mat),
                    dimnames = dimnames(p_mat))

    sig_pairs <- which(padj_mat < 0.05, arr.ind = TRUE)

    df_long <- data.frame(
        gene_A = rep(rownames(cor_mat), times = ncol(cor_mat)),
        gene_B = rep(colnames(cor_mat), each  = nrow(cor_mat)),
        correlation = as.vector(cor_mat),
        p_value     = as.vector(p_mat),
        padj        = as.vector(padj_mat)
    )

    ## Filtering based on results
    df_sig = df_long[df_long$padj<0.05,]
    print(length(df_sig$gene_A))

    out_path = paste("./Results/Revision_2/corr/",ct_1,"_",ct_2,"_cor_pair.csv",sep = "")
    write.csv(df_sig,out_path)
}

In [ ]:
cell_type = c("Astrocyte","Endothelial","Fibroblast","Microglia_Macrophage_T",
               "Mural_Cell","Neuron","Oligodendrocyte","OPC")
pairs = combn(cell_type, 2, simplify = FALSE)
# Add self-pairs manually
self_pairs = lapply(cell_type, function(x) c(x, x))
# Combine all
pairs = c(pairs, self_pairs)
pairs

In [ ]:
## run correlation analysis for all pairs
for (pair in pairs) {
    ct_1 = pair[1]
    ct_2 = pair[2]
    print(paste("Running correlation for:", ct_1, "and", ct_2))
    run_correlation(ct_1, ct_2)
}

## Draw a heatmap to show the top correlated signals across all combinations

In [ ]:
files = list.files("./Results/Revision_2/corr")

# Initialize result storage
result_list <- data.frame(
  ct1 = character(),
  ct2 = character(),
  n_sig_corr = numeric()
)

In [ ]:
cell_type = c("Astrocyte","Endothelial","Fibroblast","Microglia_Macrophage_T",
               "Mural_Cell","Neuron","Oligodendrocyte","OPC")
pairs = combn(cell_type, 2, simplify = FALSE)
# Add self-pairs manually
self_pairs = lapply(cell_type, function(x) c(x, x))
# Combine all
pairs = c(pairs, self_pairs)

In [ ]:
for (pair in pairs) {
  ct1 <- pair[1]
  ct2 <- pair[2]
  
  # Find the corresponding file for this pair (order-insensitive match)
  fn <- files[grepl(ct1, files) & grepl(ct2, files)]
  
  if (length(fn) == 1) {
    df_tmp <- read.csv(file.path("./Results/Revision_2/corr", fn))
    df_tmp_sig <- df_tmp[df_tmp$padj < 0.05 & abs(df_tmp$correlation) >= 0.75, ]
    
    # Store count
    result_list <- rbind(result_list, data.frame(
      ct1 = ct1,
      ct2 = ct2,
      n_sig_corr = nrow(df_tmp_sig)
    ))
  } else {
    message(paste("Skipping:", ct1, ct2, "- file not found or ambiguous."))
  }
}

In [ ]:
cell_type <- c("Astrocyte","Endothelial","Fibroblast","Microglia_Macrophage_T",
               "Mural_Cell","Neuron","Oligodendrocyte","OPC")

# Initialize matrix
mat <- matrix(NA, nrow = length(cell_type), ncol = length(cell_type),
              dimnames = list(cell_type, cell_type))

# Fill matrix with result_list
for (i in seq_len(nrow(result_list))) {
  ct1 <- result_list$ct1[i]
  ct2 <- result_list$ct2[i]
  val <- result_list$n_sig_corr[i]
  
  mat[ct1, ct2] <- val
  mat[ct2, ct1] <- val
}

In [ ]:
for (i in 1:length(rownames(mat))){
    fn = paste("./Results/Revision_2/corr/",rownames(mat)[i],"_",colnames(mat)[i],"_cor_pair.csv",sep = "")
    df_tmp =read.csv(fn)
    df_tmp_sig <- df_tmp[df_tmp$padj < 0.05 & abs(df_tmp$correlation) >= 0.75, ]
    mat[i,i] = nrow(df_tmp_sig)
}

In [ ]:
mat[upper.tri(mat)] <- NA
mat

In [ ]:
options(repr.plot.width = 9)
col_fun <- colorRamp2(
  breaks = c(0, 3500),
  colors = c("white", "darkred")
)
pdf("./Results/Revision_2/corr/Correlation_analysis_across_cc.pdf", width = 8, height = 6.5)
Heatmap(
  mat,
  name = "# Correlated Genes",
  col = col_fun,
  cluster_rows = FALSE,
  cluster_columns = FALSE,
  show_row_names = TRUE,
  show_column_names = TRUE,
  cell_fun = function(j, i, x, y, width, height, fill) {
    val <- mat[i, j]
    
    # Set diagonal color manually
    if (i == j) {
      grid.rect(x, y, width, height, gp = gpar(fill = "grey90", col = NA))
      grid.text(round(val), x, y, gp = gpar(fontsize = 9))
    } else if (!is.na(val)) {
      grid.text(round(val), x, y, gp = gpar(fontsize = 10))
    }
  },
  rect_gp = gpar(col = "grey90")
)
dev.off()

## Fuzzy clustering analysis

In [ ]:
cell_type <- c("Astrocyte","Endothelial","Fibroblast","Microglia_Macrophage_T",
               "Mural_Cell","Neuron","Oligodendrocyte","OPC")
mat_list = list()
for(i in 1:length(cell_type)){
    ct = cell_type[i]
    mat_temp = CreatePseudoBulk(seurat_obj = clean, target_celltype = ct, assay = "CLEAN")
    rownames(mat_temp) <- paste(rownames(mat_temp), ct,sep = "_")
    mat_list[[i]] = mat_temp
}

In [ ]:
common_samples <- Reduce(intersect, lapply(mat_list, colnames))

length(common_samples)

mat_list <- lapply(mat_list, function(mat) mat[, common_samples, drop = FALSE])


In [ ]:
vst_mat_list = lapply(mat_list, function(mat) CreateVSTMatrix(mat = mat, var.per = 0.95))
# vst_mat_list = lapply(mat_list, function(mat) CreateLogNormMatrix(mat = mat, var.per = 0.95))

In [ ]:
## Combine the list
# count_mat = Reduce(rbind, mat_list)
expr_mat = Reduce(rbind, vst_mat_list)
expr_mat = as.matrix(expr_mat)
expr_mat = expr_mat[complete.cases(expr_mat), ]
dim(expr_mat)

In [ ]:
head(expr_mat)
range(expr_mat)

In [ ]:
set.seed(42)
eset <- ExpressionSet(assayData = expr_mat)

# Standardize expression (center + scale each gene across samples)
eset_std <- standardise(eset)

m <- mestimate(eset_std)

# Visual elbow plot to choose cluster number
options(repr.plot.width = 8,repr.plot.height = 8)
Dmin(eset_std, m = m, crange = 2:30, repeats = 3, visu = TRUE)

In [ ]:
set.seed(42)
c <- 25  # chose the value based on the elbow plot above
cl <- mfuzz(eset_std, c = c, m = m)

In [ ]:
# Hard assignment: each gene_celltype assigned to 1 cluster
cluster_assignment <- data.frame(
  gene_celltype = names(cl$cluster),
  cluster = cl$cluster
)

# Fuzzy membership: probability each gene_celltype belongs to each cluster
membership <- as.data.frame(cl$membership)
membership$gene_celltype <- rownames(membership)

In [ ]:
write.csv(membership,file = "./Results/Revision_2/corr/Fuzzy_membership.csv")

In [ ]:
membership_long <- membership %>%
  pivot_longer(-gene_celltype, names_to = "cluster", values_to = "membership") %>%
  filter(membership >= 0.99)

table(membership_long$cluster)

# 2. Extract celltype from "gene_celltype"
membership_long <- membership_long %>%
  mutate(
    celltype = str_split_fixed(gene_celltype, "_", 2)[, 2],
    cluster = as.character(cluster),           # convert to character
    celltype = as.character(celltype)          # ensure celltype is not a list
  )

In [ ]:
heatmap_mat = table(membership_long$cluster,membership_long$celltype)

col_fun <- colorRamp2(c(0, max(heatmap_mat)), c("white", "darkred"))
pdf("./Results/Revision_2/corr/Fuzzy_clustering_across_cc.pdf", width = 3.5, height = 6.5)
options(repr.plot.width = 6,repr.plot.height = 6.5)
ht =Heatmap(
  heatmap_mat,
  name = "# Genes",
  col = col_fun,
  cluster_rows = T,
  cluster_columns = T,
  column_title = "Cell Class",
  row_title = "Gene Module",
  heatmap_legend_param = list(title = "# Genes")
)
draw(ht)
dev.off()

options(repr.plot.height = 15,repr.plot.width = 14)
draw(ht)

In [ ]:
cluster_id <- 10
# Get genes assigned to this cluster
cluster_members <- membership_long$gene_celltype[membership_long$cluster %in% cluster_id]

length(cluster_members)
print(table(str_split_fixed(cluster_members,"_",2)[,2]))

In [ ]:
# Subset expression matrix
expr_subset <- exprs(eset_std)[cluster_members, , drop = FALSE]
# expr_subset <- expr_mat[cluster_members, , drop = FALSE]
# expr_subset <- count_mat[cluster_members, , drop = FALSE]

df_long <- as.data.frame(expr_subset) %>%
  rownames_to_column("gene_celltype") %>%
  pivot_longer(-gene_celltype, names_to = "sample", values_to = "expression") %>%
  separate(gene_celltype, into = c("gene", "celltype"), sep = "_")%>%
  mutate(region = str_split_fixed(sample, "_", 2)[, 2])

df_avg <- df_long %>%
  group_by(gene, celltype, region) %>%
  summarise(mean_expr = mean(expression, na.rm = TRUE), .groups = "drop")

In [ ]:
col1 <- c(
  'Astrocyte' = '#F06719',
  'Neuron' = '#08415C',
  # 'Ependymal_Epithelial' = '#23767C',
  'Microglia' = '#DC143C',
  'Oligodendrocyte' = "#00BFC4",
  'Endothelial' = "#fcbe05",
  'Mural' = "#A26DC2",
  'Fibroblast' = "#5b844d",
  'OPC' = "#0072B2"
)

In [ ]:
# df_avg
df_avg$region = sub("-", "_", df_avg$region)

In [ ]:
meta = clean@meta.data
df = unique(meta[,c("region_name","brain_region","region_layer")])
# df
# df

df_avg$region_name = df[match(df_avg$region,df$brain_region),]$region_name
df_avg$order = str_split_fixed(df_avg$region,"_",n = 2)[,1]
# Arrange by custom order and extract ordered region names
df_avg$order <- as.numeric(df_avg$order)
df_avg = df_avg[order(df_avg$order),]

ordered_regions <- df_avg %>%
  arrange() %>%
  pull(region_name) %>%
  unique()

# Set factor levels for 'region' based on numeric 'order'
df_avg$region_name <- factor(df_avg$region_name, levels = ordered_regions)

In [ ]:
options(repr.plot.width = 25,repr.plot.height = 20)
p1 = ggplot(df_avg, aes(x = region_name, y = mean_expr, group = gene, color = celltype)) +
  geom_line(alpha = 0.8) +
  theme_minimal(base_size = 13) +
  scale_color_manual(values = col1) +
  theme(axis.text.x = element_text(angle = 90, hjust = 1),
        axis.text = element_text(size = 20),
        panel.border = element_rect(color = "black", fill = NA, linewidth = 1)) +
  labs(
    title = paste("Module", cluster_id, "- Gene + Cell Type Expression"),
    x = "Brain Region",
    y = "Standardized Expression",
    color = "Cell Type"
  )

In [ ]:
options(repr.plot.height = 6,repr.plot.width = 25)
# p1
p1

ggsave(filename = "./Results/Revision_2/corr/Fuzzy_module_10.pdf", 
patchwork::wrap_plots(p1, ncol = 1),
scale = 1, width = 14, height = 8.5)

## Run enrichment analysis to show the functions of the genes in one cell type of one module

In [ ]:
suppressPackageStartupMessages({
library(clusterProfiler)
library('org.Hs.eg.db')
})

In [ ]:
## chossing the module
cluster_id <- 2

In [ ]:
# Get genes assigned to this cluster
cluster_members = data.frame(member = membership_long$gene_celltype[membership_long$cluster %in% cluster_id])
cluster_members$gene = str_split_fixed(cluster_members$member,"_",2)[,1]
cluster_members$cell.type = str_split_fixed(cluster_members$member,"_",2)[,2]

## adding gene id
cluster_members$gene_id <- mapIds(
  # Replace with annotation package for the organism relevant to your data
  org.Hs.eg.db,
  # The vector of gene identifiers we want to map
  keys = cluster_members$gene,
  # Replace with the type of gene identifiers in your data
  keytype = "SYMBOL",
  # Replace with the type of gene identifiers you would like to map to
  column = "ENTREZID",
  # In the case of 1:many mappings, return the
  # first one. This is default behavior!
  multiVals = "first"
)


cluster_members = cluster_members[cluster_members$cell.type %in% names(which(table(cluster_members$cell.type) > 5)),]

head(cluster_members)
table(cluster_members$cell.type)

df = data.frame()
for(i in unique(cluster_members$cell.type)){
    ego <- enrichGO(gene = cluster_members[cluster_members$cell.type == i,]$gene_id,
                OrgDb = org.Hs.eg.db,
                ont = c("BP"),
                pAdjustMethod = "BH",
                pvalueCutoff  = 0.05,
                qvalueCutoff  = 0.05,
                readable = TRUE)
    if (is.null(ego)){
        rint(paste("No significant results for",i))
    }else{
        ego <- simplify(ego) 
        df_temp = ego@result
        ifelse(nrow(df_temp) > 0,df_temp$cluster <- i,df_temp$cluster <- character())
        df = rbind(df,df_temp)
    }
}

In [ ]:
top <- df %>% group_by(cluster) %>% top_n(n = 5, wt = -p.adjust) %>% 
        mutate(log_padj = -log10(p.adjust),
         Description = factor(Description, levels = unique(Description)))
table(top$cluster)
# top$Description <- factor(top$Description, levels = unique(top$Description))

In [ ]:
unique(top$cluster)

col2 <- c(
  'Astrocyte' = '#F06719',
  'Neuron' = '#08415C',
  # 'Ependymal_Epithelial' = '#23767C',
  'Microglia_Macrophage_T' = '#DC143C',
  'Oligodendrocyte' = "#00BFC4",
  'Endothelial' = "#fcbe05",
  'Mural_Cell' = "#A26DC2",
  'Fibroblast' = "#5b844d",
)


In [ ]:
ct = "Endothelial"
top1 = top[top$cluster == ct,]
p1 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Mural_Cell"
top1 = top[top$cluster == ct,]
p2 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

In [ ]:
options(repr.plot.height = 5,repr.plot.width =35)
p1 + p2

ggsave(filename = "./Results/Revision_2/corr/Module_2_GOBP.PDF", 
patchwork::wrap_plots(p1,p2, ncol = 1),
scale = 1, width = 13, height = 10)

In [ ]:
ct = "Endothelial"
top1 = top[top$cluster == ct,]
p1 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Astrocyte"
top1 = top[top$cluster == ct,]
p2 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Fibroblast"
top1 = top[top$cluster == ct,]
p3 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text = element_text(size = 20),
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Mural_Cell"
top1 = top[top$cluster == ct,]
p4 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text = element_text(size = 20),
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

In [ ]:
options(repr.plot.height = 5,repr.plot.width =35)
p1 + p2 + p3 + p4

ggsave(filename = "./Results/Revision_2/corr/Module_12_GOBP.PDF", 
patchwork::wrap_plots(p1,p2,p3,p4, ncol = 1),
scale = 1, width = 13, height = 15)

In [ ]:
ct = "Endothelial"
top1 = top[top$cluster == ct,]
p1 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Mural_Cell"
top1 = top[top$cluster == ct,]
p2 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

ct = "Fibroblast"
top1 = top[top$cluster == ct,]
p3 = ggplot(top1, aes(x = `log_padj`, y = reorder(Description, `log_padj`))) +
      geom_bar(stat = "identity", fill = col2[ct]) +
    #   theme_minimal() +
      xlab("-log10(Adjusted p-value)") +
      ylab("GO Term") +
      theme(
        axis.text = element_text(size = 20),
        axis.text.x = element_text(size = 20),
        axis.text.y = element_text(size = 20,),
        panel.border = element_rect(color = "black", fill = NA, size = 1),  # Add border
        plot.margin = margin(10, 10, 10, 10),
        panel.background = element_blank() 
        ) 

In [ ]:
options(repr.plot.height = 5,repr.plot.width =35)
p1 + p2 + p3

ggsave(filename = "./Results/Revision_2/corr/Module_10_GOBP.PDF", 
patchwork::wrap_plots(p1,p2,p3, ncol = 1),
scale = 1, width = 10, height = 15)